# DeepAR Demand Forecasting

## Objective

The objective of this notebook is to generate final DeepAR forecasts
using the pre-trained model for the project's blind 15-day validation period.

DeepAR was trained as a global probabilistic forecasting model across
Store × Family time series.

## Note

This notebook uses the pre-trained DeepAR model saved in the project's
`models/` directory.

No retraining is performed in this notebook.

The notebook only:
- rebuilds the final forecasting dataset,
- loads the trained checkpoint,
- generates validation predictions,
- evaluates the predictions using the shared evaluation framework,
- saves the results, and
- updates the combined prediction dataset.

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import os
import sys
import numpy as np
import pandas as pd
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.4.1+cpu
CUDA available: False


In [2]:
# ============================================================
# SETUP PROJECT ROOT + IMPORT EVALUATION
# ============================================================

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from evaluation import (
    FORECAST_HORIZON,
    filter_evaluation_data,
    evaluate
)

print("✅ Setup complete")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FORECAST_HORIZON:", FORECAST_HORIZON)

✅ Setup complete
PROJECT_ROOT: c:\Users\ino0i\OneDrive\سطح المكتب\Retail_Demand_Forec
FORECAST_HORIZON: 15


In [3]:
# ============================================================
# LOAD DATA + FILTER EVALUATION
# ============================================================

train_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "train_df.parquet"
)

valid_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "valid_df.parquet"
)

train_df = pd.read_parquet(train_path)
valid_df = pd.read_parquet(valid_path)

valid_df_eval = filter_evaluation_data(valid_df)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Valid eval shape:", valid_df_eval.shape)
print()

print(
    "Date range:",
    valid_df_eval["date"].min(),
    "→",
    valid_df_eval["date"].max()
)

print(
    "Unique series:",
    valid_df_eval.groupby(
        ["store_nbr", "family"]
    ).ngroups
)

Train shape: (2364490, 23)
Valid shape: (25929, 23)
Valid eval shape: (25920, 23)

Date range: 2017-08-01 00:00:00 → 2017-08-15 00:00:00
Unique series: 1728


In [4]:
# ============================================================
# COMBINE TRAIN + VALID
# ============================================================

keep_cols = [
    "date",
    "store_nbr",
    "family",
    "sales",
    "onpromotion",
    "weekday",
    "month"
]

train_small = train_df[keep_cols].copy()
valid_small = valid_df_eval[keep_cols].copy()

train_small["is_train"] = True
valid_small["is_train"] = False

df_all = pd.concat(
    [train_small, valid_small],
    ignore_index=True
)

df_all = (
    df_all
    .sort_values(["store_nbr", "family", "date"])
    .reset_index(drop=True)
)

df_all["series_id"] = (
    df_all["store_nbr"].astype(str)
    + "_"
    + df_all["family"].astype(str)
)

df_all["time_idx"] = (
    df_all["date"] - df_all["date"].min()
).dt.days

df_all["store_nbr"] = df_all["store_nbr"].astype(str)
df_all["family"] = df_all["family"].astype(str)
df_all["weekday"] = df_all["weekday"].astype(str)
df_all["month"] = df_all["month"].astype(str)

print("df_all shape:", df_all.shape)
print(
    "time_idx range:",
    df_all["time_idx"].min(),
    "→",
    df_all["time_idx"].max()
)

df_all shape: (2390410, 10)
time_idx range: 0 → 1658


In [5]:
# ============================================================
# REBUILD training_full + FINAL EVALUATION CONTEXT
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

MAX_ENCODER_LENGTH = 30
MAX_PREDICTION_LENGTH = 15

train_data_full = df_all[
    df_all["is_train"] == True
].copy()

final_eval = df_all[
    df_all["is_train"] == False
].copy()

training_full = TimeSeriesDataSet(
    train_data_full,
    time_idx="time_idx",
    target="sales",
    group_ids=["series_id"],

    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,

    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    static_categoricals=[
        "store_nbr",
        "family"
    ],

    time_varying_known_categoricals=[
        "weekday",
        "month"
    ],

    time_varying_known_reals=[
        "onpromotion"
    ],

    time_varying_unknown_reals=[
        "sales"
    ],

    target_normalizer=GroupNormalizer(
        groups=["series_id"],
        transformation="log1p"
    ),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

train_data_full_max_time_idx = (
    train_data_full["time_idx"].max()
)

encoder_history_final = train_data_full[
    train_data_full["time_idx"]
    > (train_data_full_max_time_idx - MAX_ENCODER_LENGTH)
].copy()

final_eval_with_context = pd.concat(
    [
        encoder_history_final,
        final_eval
    ],
    ignore_index=True
)

final_eval_with_context = (
    final_eval_with_context
    .sort_values(["series_id", "time_idx"])
    .reset_index(drop=True)
)

print("✅ training_full samples:", len(training_full))
print(
    "✅ encoder_history_final shape:",
    encoder_history_final.shape
)
print(
    "✅ final_eval_with_context shape:",
    final_eval_with_context.shape
)

✅ training_full samples: 2400281
✅ encoder_history_final shape: (51840, 10)
✅ final_eval_with_context shape: (77760, 10)


In [6]:
import os
import torch

checkpoint_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "deepar_full_trained.ckpt"
)

print("Checkpoint:", checkpoint_path)
print("Exists:", os.path.exists(checkpoint_path))

checkpoint = torch.load(
    checkpoint_path,
    map_location=torch.device("cpu"),
    weights_only=False
)

print("\nCheckpoint loaded!")
print("Keys:", checkpoint.keys())

Checkpoint: c:\Users\ino0i\OneDrive\سطح المكتب\Retail_Demand_Forec\models\deepar_full_trained.ckpt
Exists: True

Checkpoint loaded!
Keys: dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters', 'dataset_parameters', '__special_save__'])


In [7]:
hp = checkpoint.get("hyper_parameters", {})

print("Number of hyperparameters:", len(hp))

for key in sorted(hp.keys()):
    print(f"{key}: {hp[key]}")

Number of hyperparameters: 35
categorical_groups: {}
cell_type: LSTM
dataset_parameters: {'time_idx': 'time_idx', 'target': 'sales', 'group_ids': ['series_id'], 'weight': None, 'max_encoder_length': 30, 'min_encoder_length': 15, 'min_prediction_idx': 0, 'min_prediction_length': 1, 'max_prediction_length': 15, 'static_categoricals': ['store_nbr', 'family'], 'static_reals': None, 'time_varying_known_categoricals': ['weekday', 'month'], 'time_varying_known_reals': ['onpromotion'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['sales'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': True, 'lags': None, 'add_relative_time_idx': True, 'add_target_scales': True, 'add_encoder_length': True, 'target_normalizer': GroupNormalizer(
	method='standard',
	groups=['series_id'],
	center=True,
	scale_by_group=False,
	transformation='log1p',
	method_kwargs={}
), 'categorical_encoders': {'__group_id__series_id': NaNLabelEncoder(add_nan=False

In [8]:
state_dict = checkpoint["state_dict"]

print("Number of parameters:", len(state_dict))

for key in list(state_dict.keys())[:50]:
    print(key, state_dict[key].shape)

Number of parameters: 14
embeddings.embeddings.store_nbr.weight torch.Size([54, 15])
embeddings.embeddings.family.weight torch.Size([33, 11])
embeddings.embeddings.weekday.weight torch.Size([7, 5])
embeddings.embeddings.month.weight torch.Size([12, 6])
rnn.weight_ih_l0 torch.Size([128, 43])
rnn.weight_hh_l0 torch.Size([128, 32])
rnn.bias_ih_l0 torch.Size([128])
rnn.bias_hh_l0 torch.Size([128])
rnn.weight_ih_l1 torch.Size([128, 32])
rnn.weight_hh_l1 torch.Size([128, 32])
rnn.bias_ih_l1 torch.Size([128])
rnn.bias_hh_l1 torch.Size([128])
distribution_projector.weight torch.Size([2, 32])
distribution_projector.bias torch.Size([2])


In [9]:
from pytorch_forecasting.models import DeepAR

model = DeepAR.from_dataset(
    training_full,
    hidden_size=32,
    rnn_layers=2,
    dropout=0.1,
    learning_rate=1e-3,
)

print("Model created.")

model.load_state_dict(
    checkpoint["state_dict"],
    strict=True
)

print("Weights loaded successfully!")

model = model.cpu()
model.eval()

print("DeepAR is ready on CPU.")

Model created.
Weights loaded successfully!
DeepAR is ready on CPU.


In [10]:
predictions = model.predict(
    val_dataloader_final_full,
    mode="prediction",
    return_x=True,
)

NameError: name 'val_dataloader_final_full' is not defined

In [ ]:
pred_raw = predictions.output.detach().cpu().numpy()

pred_final = np.clip(
    pred_raw,
    0,
    None
)

print("Prediction shape:", pred_final.shape)
print("Min:", pred_final.min())
print("Max:", pred_final.max())

Prediction shape: (1728, 15)
Min: 0.0
Max: 887820.5


In [ ]:
prediction_result = model.predict(
    val_dataloader_final_full,
    mode="prediction",
    return_index=True,
    return_x=False,
)

pred_raw = prediction_result.output.detach().cpu().numpy()
pred_final = np.clip(pred_raw, 0, None)

print("Prediction shape:", pred_final.shape)
print("Prediction min:", pred_final.min())
print("Prediction max:", pred_final.max())

prediction_index = prediction_result.index.copy()

print(prediction_index.head(10))
print("Number of series:", len(prediction_index))

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Prediction shape: (1728, 15)
Prediction min: 0.0
Prediction max: 299301.78
   time_idx        series_id
0      1644    10_AUTOMOTIVE
1      1644     10_BABY CARE
2      1644        10_BEAUTY
3      1644     10_BEVERAGES
4      1644  10_BREAD/BAKERY
5      1644   10_CELEBRATION
6      1644      10_CLEANING
7      1644         10_DAIRY
8      1644          10_DELI
9      1644          10_EGGS
Number of series: 1728


In [ ]:
# Prepare evaluation validation data
valid_eval = filter_evaluation_data(valid_df).copy()

# Get validation dates
validation_dates = sorted(valid_eval["date"].unique())

print("Number of validation dates:", len(validation_dates))
print("First dates:", validation_dates[:3])
print("Last dates:", validation_dates[-3:])

Number of validation dates: 15
First dates: [Timestamp('2017-08-01 00:00:00'), Timestamp('2017-08-02 00:00:00'), Timestamp('2017-08-03 00:00:00')]
Last dates: [Timestamp('2017-08-13 00:00:00'), Timestamp('2017-08-14 00:00:00'), Timestamp('2017-08-15 00:00:00')]


In [ ]:
# Copy prediction index
prediction_index = prediction_result.index.copy()

# Extract store number and product family
prediction_index["store_nbr"] = (
    prediction_index["series_id"]
    .str.split("_", n=1)
    .str[0]
    .astype(int)
)

prediction_index["family"] = (
    prediction_index["series_id"]
    .str.split("_", n=1)
    .str[1]
)

# Repeat each series for the 15 forecast days
deepar_predictions = prediction_index[
    ["store_nbr", "family"]
].loc[
    prediction_index.index.repeat(FORECAST_HORIZON)
].reset_index(drop=True)

# Add the 15 validation dates to every series
deepar_predictions["date"] = np.tile(
    validation_dates,
    len(prediction_index)
)

# Flatten (1728, 15) -> 25,920 predictions
deepar_predictions["deepar_pred"] = pred_final.reshape(-1)

# Final column order
deepar_predictions = deepar_predictions[
    ["date", "store_nbr", "family", "deepar_pred"]
]

print("Shape:", deepar_predictions.shape)
print(
    "Unique series:",
    deepar_predictions[["store_nbr", "family"]]
    .drop_duplicates()
    .shape[0]
)
print("Unique dates:", deepar_predictions["date"].nunique())

deepar_predictions.head(20)

Shape: (25920, 4)
Unique series: 1728
Unique dates: 15


,date,store_nbr,family,deepar_pred
0,2017-08-01,10,AUTOMOTIVE,2.107015
1,2017-08-02,10,AUTOMOTIVE,1.372926
2,2017-08-03,10,AUTOMOTIVE,1.640650
3,2017-08-04,10,AUTOMOTIVE,1.514944
4,2017-08-05,10,AUTOMOTIVE,2.159912
5,2017-08-06,10,AUTOMOTIVE,2.177983
6,2017-08-07,10,AUTOMOTIVE,1.605833
7,2017-08-08,10,AUTOMOTIVE,1.804890
8,2017-08-09,10,AUTOMOTIVE,1.411808
9,2017-08-10,10,AUTOMOTIVE,1.350923


In [ ]:
actuals = valid_eval[
    ["date", "store_nbr", "family", "sales"]
].copy()

deepar_eval = deepar_predictions.merge(
    actuals,
    on=["date", "store_nbr", "family"],
    how="left",
    validate="one_to_one"
)

print("Shape:", deepar_eval.shape)
print("Missing actual sales:", deepar_eval["sales"].isna().sum())
print("Missing predictions:", deepar_eval["deepar_pred"].isna().sum())

deepar_eval.head()

Shape: (25920, 5)
Missing actual sales: 0
Missing predictions: 0


,date,store_nbr,family,deepar_pred,sales
0,2017-08-01,10,AUTOMOTIVE,2.107015,1.0
1,2017-08-02,10,AUTOMOTIVE,1.372926,0.0
2,2017-08-03,10,AUTOMOTIVE,1.640650,2.0
3,2017-08-04,10,AUTOMOTIVE,1.514944,4.0
4,2017-08-05,10,AUTOMOTIVE,2.159912,2.0


In [ ]:
deepar_eval["prediction_error"] = (
    deepar_eval["deepar_pred"] - deepar_eval["sales"]
)

top_predictions = deepar_eval.nlargest(
    20,
    "deepar_pred"
)[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "deepar_pred",
        "prediction_error"
    ]
]

top_predictions

,date,store_nbr,family,sales,deepar_pred,prediction_error
8580,2017-08-01,26,PRODUCE,591.653,299301.781250,298710.128250
5190,2017-08-01,1,PRODUCE,2560.610,114995.992188,112435.382187
18465,2017-08-01,45,HOME CARE,816.000,71544.953125,70728.953125
14295,2017-08-01,37,PRODUCE,2337.738,49004.878906,46667.140906
19605,2017-08-01,47,PRODUCE,5693.949,47490.929688,41796.980687
19125,2017-08-01,46,PRODUCE,2734.955,45441.128906,42706.173906
20565,2017-08-01,49,PRODUCE,8002.368,25836.490234,17834.122234
18645,2017-08-01,45,PRODUCE,5768.652,19076.587891,13307.935891
8085,2017-08-01,25,PRODUCE,1265.909,18946.306641,17680.397641
15765,2017-08-01,3,PRODUCE,7382.047,16655.585938,9273.538938


In [ ]:
pattern_lookup = valid_eval[
    ["store_nbr", "family", "pattern"]
].drop_duplicates()

deepar_eval = deepar_eval.merge(
    pattern_lookup,
    on=["store_nbr", "family"],
    how="left",
    validate="many_to_one"
)

print(deepar_eval["pattern"].value_counts())

pattern
Regular         16920
Irregular        5670
Intermittent     3330
Name: count, dtype: int64


In [ ]:
deepar_metrics = evaluate(
    deepar_eval["sales"],
    deepar_eval["deepar_pred"],
    name="DeepAR"
)

deepar_metrics

DeepAR               MAE:   107.49 | RMSE:  2114.82 | RMSLE: 0.4872 | WMAPE:  22.41%


{'Model': 'DeepAR',
 'MAE': 107.48676006434381,
 'RMSE': np.float64(2114.82091596113),
 'RMSLE': np.float64(0.48719465139394896),
 'WMAPE': np.float64(22.407982436701428)}

In [ ]:
pattern_results = []

for pattern_name, group in deepar_eval.groupby("pattern"):
    metrics = evaluate(
        group["sales"],
        group["deepar_pred"],
        name="DeepAR"
    )
    
    metrics["Pattern"] = pattern_name
    metrics["N"] = len(group)
    
    pattern_results.append(metrics)

deepar_pattern_metrics = pd.DataFrame(pattern_results)

deepar_pattern_metrics

DeepAR               MAE:     0.96 | RMSE:     3.85 | RMSLE: 0.5144 | WMAPE:  93.01%
DeepAR               MAE:   105.20 | RMSE:  4085.99 | RMSLE: 0.7039 | WMAPE:  92.58%
DeepAR               MAE:   129.22 | RMSE:  1121.04 | RMSLE: 0.3814 | WMAPE:  18.55%


,Model,MAE,RMSE,RMSLE,WMAPE,Pattern,N
0,DeepAR,0.955752,3.850225,0.514360,93.005669,Intermittent,3330
1,DeepAR,105.199739,4085.990348,0.703949,92.581202,Irregular,5670
2,DeepAR,129.219364,1121.039399,0.381424,18.551349,Regular,16920


In [ ]:
from pathlib import Path

# Find the project root
current_path = Path.cwd()

print("Current working directory:")
print(current_path)

Current working directory:
c:\Users\ino0i\OneDrive\سطح المكتب\Retail_Demand_Forec\notebooks


In [ ]:
project_root = current_path.parent

print("Project root:")
print(project_root)

Project root:
c:\Users\ino0i\OneDrive\سطح المكتب\Retail_Demand_Forec


In [ ]:
prediction_output_path = (
    project_root
    / "predictions"
    / "deepar_predictions.parquet"
)

prediction_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

deepar_eval.to_parquet(
    prediction_output_path,
    index=False
)

print("Saved:")
print(prediction_output_path)

Saved:
c:\Users\ino0i\OneDrive\سطح المكتب\Retail_Demand_Forec\predictions\deepar_predictions.parquet


In [ ]:
master_path = (
    project_root
    / "data"
    / "processed"
    / "prediction_results.parquet"
)

master_predictions = pd.read_parquet(master_path)

print("Master shape:", master_predictions.shape)
print("Columns:")
print(master_predictions.columns.tolist())

Master shape: (25920, 8)
Columns:
['date', 'store_nbr', 'family', 'pattern', 'sales', 'seasonal_naive_pred', 'xgboost_pred', 'lightgbm_pred']


In [ ]:
master_predictions = master_predictions.drop(
    columns=["deepar_pred"],
    errors="ignore"
)

master_predictions = master_predictions.merge(
    deepar_predictions,
    on=["date", "store_nbr", "family"],
    how="left",
    validate="one_to_one"
)

print("Updated shape:", master_predictions.shape)
print(
    "Missing DeepAR predictions:",
    master_predictions["deepar_pred"].isna().sum()
)

Updated shape: (25920, 9)
Missing DeepAR predictions: 0


In [ ]:
model_columns = [
    "seasonal_naive_pred",
    "xgboost_pred",
    "lightgbm_pred",
    "deepar_pred"
]

print("Model columns:")
for col in model_columns:
    print(
        col,
        "→",
        "OK" if col in master_predictions.columns else "MISSING"
    )

Model columns:
seasonal_naive_pred → OK
xgboost_pred → OK
lightgbm_pred → OK
deepar_pred → OK


In [ ]:
print("Rows:", len(master_predictions))

print(
    "Unique series:",
    master_predictions[
        ["store_nbr", "family"]
    ].drop_duplicates().shape[0]
)

print(
    "Unique dates:",
    master_predictions["date"].nunique()
)

print(
    "Missing model predictions:"
)

for col in model_columns:
    print(
        col,
        "→",
        master_predictions[col].isna().sum()
    )

Rows: 25920
Unique series: 1728
Unique dates: 15
Missing model predictions:
seasonal_naive_pred → 0
xgboost_pred → 0
lightgbm_pred → 0
deepar_pred → 0


In [ ]:
master_predictions.to_parquet(
    master_path,
    index=False
)

print("Master prediction file updated successfully.")

Master prediction file updated successfully.


In [ ]:
print("========== DEEPAR CLEAN COMPLETE CHECK ==========")

print("DeepAR prediction rows:", len(deepar_predictions))
print(
    "DeepAR unique series:",
    deepar_predictions[
        ["store_nbr", "family"]
    ].drop_duplicates().shape[0]
)
print(
    "DeepAR unique dates:",
    deepar_predictions["date"].nunique()
)
print(
    "DeepAR missing:",
    deepar_predictions["deepar_pred"].isna().sum()
)

print("\n========== MASTER FILE ==========")

print("Master rows:", len(master_predictions))

for col in model_columns:
    print(
        f"{col}:",
        master_predictions[col].isna().sum(),
        "missing"
    )

========== DEEPAR CLEAN COMPLETE CHECK ==========
DeepAR prediction rows: 25920
DeepAR unique series: 1728
DeepAR unique dates: 15
DeepAR missing: 0

========== MASTER FILE ==========
Master rows: 25920
seasonal_naive_pred: 0 missing
xgboost_pred: 0 missing
lightgbm_pred: 0 missing
deepar_pred: 0 missing
